# Ingesta RAG con Gemini y MongoDB Atlas

Notebook para Colab. Usa el PDF indicado en la guía y guarda las credenciales en el panel **Secrets** de Colab. No escribas claves directamente en celdas.

In [1]:
!pip -q install pymongo google-genai pypdf



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from google.colab import userdata, files
from google import genai
from google.genai import types
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from pypdf import PdfReader
from pathlib import Path
from datetime import datetime, timezone
import hashlib, time

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
MONGODB_URI = userdata.get('MONGODB_URI')
assert GOOGLE_API_KEY, 'Falta GOOGLE_API_KEY en Colab Secrets'
assert MONGODB_URI, 'Falta MONGODB_URI en Colab Secrets'

DB_NAME = 'pdf_embeddings_db'
COLLECTION_NAME = 'pdf_vectors'
INDEX_NAME = 'vector_index'
EMBEDDING_MODEL = 'gemini-embedding-001'
EMBEDDING_DIMENSIONS = 3072
CHUNK_SIZE = 1000
BATCH_SIZE = 16

genai_client = genai.Client(api_key=GOOGLE_API_KEY)
mongo_client = MongoClient(
    MONGODB_URI,
    serverSelectionTimeoutMS=10000
)

# Verificar conexión con MongoDB Atlas
mongo_client.admin.command('ping')

# Obtener la base de datos
database = mongo_client[DB_NAME]

# Crear la colección si todavía no existe
if COLLECTION_NAME not in database.list_collection_names():
    database.create_collection(COLLECTION_NAME)
    print(f'Colección creada: {COLLECTION_NAME}')
else:
    print(f'Colección existente: {COLLECTION_NAME}')

# Obtener la colección
collection = database[COLLECTION_NAME]


## 1. Cargar el PDF
Sube `aws-cloud-adoption-framework_XL.pdf` desde Canvas cuando ejecutes la actividad principal.

In [ ]:
uploaded = files.upload()
pdf_path = Path(next(iter(uploaded.keys())))
reader = PdfReader(str(pdf_path))
page_texts = []
for page_number, page in enumerate(reader.pages, start=1):
    extracted = (page.extract_text() or '').strip()
    if extracted:
        page_texts.append(extracted)
    else:
        print(f'Advertencia: página {page_number} sin texto extraíble; puede requerir OCR.')
texto = '\n'.join(page_texts).strip()
assert texto, 'No se pudo extraer texto del PDF.'
print('Caracteres extraídos:', len(texto))

## 2. Fragmentación en chunks de 1000 caracteres

In [ ]:
trozos = [texto[i:i+CHUNK_SIZE] for i in range(0, len(texto), CHUNK_SIZE) if texto[i:i+CHUNK_SIZE].strip()]
print('Cantidad de fragmentos:', len(trozos))
print('Longitud del primero:', len(trozos[0]))

## 3. Crear el índice vectorial
La definición respeta la guía: tipo `vectorSearch`, campo `embedding`, 3072 dimensiones y similitud `dotProduct`. Atlas construye el índice de forma asíncrona.

In [ ]:
index_model = SearchIndexModel(
    definition={'fields': [{'type': 'vector', 'path': 'embedding', 'numDimensions': EMBEDDING_DIMENSIONS, 'similarity': 'dotProduct'}]},
    name=INDEX_NAME,
    type='vectorSearch',
)
existing = list(collection.list_search_indexes())
if not any(item.get('name') == INDEX_NAME for item in existing):
    collection.create_search_index(model=index_model)
    print('Solicitud de creación enviada:', INDEX_NAME)
else:
    print('El índice ya existe:', INDEX_NAME)


## 4. Generar embeddings e insertar fragmentos
Se usa `RETRIEVAL_DOCUMENT` para vectorizar los fragmentos del PDF. El notebook reemplaza la ingesta anterior de ese mismo nombre de archivo para evitar duplicados.

In [ ]:
sha256 = hashlib.sha256(pdf_path.read_bytes()).hexdigest()
collection.delete_many({'source': pdf_path.name})
documentos = []
now = datetime.now(timezone.utc)
for start in range(0, len(trozos), BATCH_SIZE):
    batch = trozos[start:start+BATCH_SIZE]
    response = genai_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=batch,
        config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT'),
    )
    for offset, item in enumerate(response.embeddings):
        vector = [float(v) for v in item.values]
        assert len(vector) == EMBEDDING_DIMENSIONS, f'Dimensión inesperada: {len(vector)}'
        chunk_id = start + offset
        documentos.append({'id': chunk_id, 'texto': batch[offset], 'embedding': vector, 'source': pdf_path.name, 'source_sha256': sha256, 'ingested_at': now})
    print(f'{min(start+len(batch), len(trozos))}/{len(trozos)} embeddings')
if documentos:
    collection.insert_many(documentos, ordered=True)
print('Insertados:', len(documentos))
print('Documentos en la colección:', collection.count_documents({}))

## 5. Verificación
En MongoDB Atlas abre **Database → Browse Collections** para revisar los documentos y **Search & Vector Search** para confirmar que `vector_index` termine en estado `READY`. No continúes con el chatbot mientras el índice siga construyéndose.

In [ ]:
print('Colección:', DB_NAME + '.' + COLLECTION_NAME)
print('Índices visibles:', list(collection.list_search_indexes()))

## Tarea extra: usar otra colección
Para la segunda aplicación cambia `COLLECTION_NAME = 'actividad2_vectors'` e `INDEX_NAME = 'actividad2_vector_index'`. Mantén la base `pdf_embeddings_db` si quieres reutilizar el clúster, pero no reutilices la colección ni el índice de la actividad principal. Sube el segundo PDF y vuelve a ejecutar las celdas de ingesta.